# 5. 스포츠로만 배운 모델에 날씨를 넣으면? — 톤이 따라올까
내 모델은 **스포츠 리포트 → 앵커 리드**만 배웠어요. 한 번도 본 적 없는 **날씨**를 넣어도 그 형식이 나올까요?

| 부분 | 넣는 것 | 정답 | 보는 점 |
|---|---|---|---|
| ① 날씨 리포트 | MBC 날씨 리포트 5건 (`test_weather.jsonl`) | 실제 앵커 리드 | 주제가 바뀌어도 앵커 리드 형식이 나오나 |
| ② 긴 글 지시 | 같은 5건 + "700자 리포트를 써라" | 없음 | 지시가 바뀌어도 배운 습관(짧게)이 남나 |
| ③ 날씨 데이터 | Open-Meteo 내일 예보 숫자 | 없음 | 숫자만 줘도 쓰나, 해석을 지어내지 않나 |

준비: 4번과 같아요 + `test_weather.jsonl`을 이 폴더에 넣기 (기사 원문이라 GitHub에는 없어요. 강사에게 받으세요)

In [1]:
# ▶ 1. 설정 — 여기만 고치세요
import os
import pandas as pd
from dotenv import load_dotenv
from compare_tools import connect, ask_models, auto_summary, make_report

MY_MODEL = 'mbcsports'      # ollama list 에 나온 내 모델 이름
load_dotenv()               # 같은 폴더의 .env 에서 OPENAI_API_KEY 읽기
MODELS = connect(MY_MODEL, os.getenv('OPENAI_API_KEY'))
pd.set_option('display.max_colwidth', None)
print('API 키:', '준비됨' if os.getenv('OPENAI_API_KEY') else '없음 → gpt-4.1-mini 칸은 (키 없음)')
import requests

API 키: 준비됨


## ① 날씨 리포트 5건 → 앵커 리드
학습 때와 같은 과제예요. 지시문의 "스포츠"만 "날씨"로 바꿨어요.
**먼저 주황 상자(실제 앵커 리드)를 읽고** 세 모델을 보면 차이가 잘 보여요.

In [2]:
weather = pd.read_json('test_weather.jsonl', lines=True)
weather['instruction'] = '날씨 리포트 본문을 읽고, 앵커가 읽을 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'
ans_lead = ask_models(MODELS, weather, 'weather_lead')
display(auto_summary(weather, ans_lead))
make_report(weather, ans_lead, 'weather_report.html', '스포츠로 배운 모델 × 날씨 리포트',
            note='볼 곳 ① 끝 문장: "○○○ 기자입니다"(앵커가 넘김) vs "MBC뉴스 ○○○입니다"(기자 끝인사) '
                 '② 길이: 정답과 비슷한가 ③ 말투: "~는데요"', gold_label='실제 앵커 리드')

묻는 중: 원본 Qwen2.5-3B (5문제)
묻는 중: 내가 학습한 모델 (5문제)
묻는 중: gpt-4.1-mini (5문제)
✅ 기록 저장: results\weather_lead_20261112_1559.json


,답한 문제,끝인사 없음,기자에게 넘김,2문장 이하,길이 비슷(0.5~2배),입력에 없는 숫자 없음,평균 글자 수
모델,,,,,,,
원본 Qwen2.5-3B,5/5,3/5,1/5,0/5,0/5,5/5,526
내가 학습한 모델,5/5,5/5,3/5,1/5,5/5,5/5,105
gpt-4.1-mini,5/5,5/5,2/5,0/5,4/5,4/5,189


✅ 비교표 완성: C:\workspace\ai-datajournalism-labs\4_파인튜닝\weather_report.html


## ② 지시문만 바꿔 보기 — "700자 리포트를 써라"
GPT는 지시대로 길게 쓸 거예요. 내 모델은? **짧게, "○○○ 기자입니다"로 끝내는 습관**이 남는다면 파인튜닝이 지시문보다 강하게 작동한 거예요.

In [3]:
weather_long = weather.copy()
weather_long['instruction'] = '날씨 리포트 본문을 읽고, MBC 뉴스 기자 리포트 기사를 700자 이상으로 쓰세요.'
ans_long = ask_models(MODELS, weather_long, 'weather_long')
display(auto_summary(weather_long, ans_long))
make_report(weather_long, ans_long, 'weather_long_report.html', '지시문을 바꾸면? 700자 리포트 지시',
            note='평균 글자 수를 ①과 비교해 보세요. 내 모델이 여전히 짧다면 학습한 습관이 남은 거예요.',
            gold_col='-')

묻는 중: 원본 Qwen2.5-3B (5문제)
묻는 중: 내가 학습한 모델 (5문제)
묻는 중: gpt-4.1-mini (5문제)
✅ 기록 저장: results\weather_long_20261112_1600.json


,답한 문제,끝인사 없음,기자에게 넘김,2문장 이하,길이 비슷(0.5~2배),입력에 없는 숫자 없음,평균 글자 수
모델,,,,,,,
원본 Qwen2.5-3B,5/5,5/5,0/5,0/5,0/5,4/5,779
내가 학습한 모델,5/5,5/5,1/5,2/5,4/5,4/5,168
gpt-4.1-mini,5/5,0/5,1/5,0/5,0/5,5/5,1158


✅ 비교표 완성: C:\workspace\ai-datajournalism-labs\4_파인튜닝\weather_long_report.html


## ③ Open-Meteo 날씨 데이터 → 앵커 리드
[Open-Meteo](https://open-meteo.com)는 가입 없이 쓰는 무료 날씨 API예요. 내일 예보 **숫자**만 메모로 만들어 넣어요.
정답이 없으니 **메모와 대조**가 핵심이에요. 숫자 배지가 초록이어도 해석("강수확률이 높다")이나 숫자의 자리가 틀릴 수 있어요.

In [4]:
CITIES = {'서울': (37.5665, 126.9780), '강릉': (37.7519, 128.8761), '대전': (36.3504, 127.3845),
          '부산': (35.1796, 129.0756), '제주': (33.4996, 126.5312)}
WMO = {0: '맑음', 1: '대체로 맑음', 2: '구름 조금', 3: '흐림', 45: '안개', 48: '안개', 51: '이슬비', 53: '이슬비', 55: '이슬비',
       61: '비', 63: '비', 65: '강한 비', 71: '눈', 73: '눈', 75: '강한 눈', 80: '소나기', 81: '소나기', 82: '강한 소나기', 95: '뇌우'}

def forecast(lat, lon):
    p = {'latitude': lat, 'longitude': lon, 'timezone': 'Asia/Seoul', 'forecast_days': 2, 'wind_speed_unit': 'ms',
         'daily': 'weather_code,temperature_2m_max,temperature_2m_min,precipitation_probability_max,precipitation_sum,wind_speed_10m_max'}
    return requests.get('https://api.open-meteo.com/v1/forecast', params=p, timeout=15).json()['daily']

rows = []
for city, (lat, lon) in CITIES.items():
    d = forecast(lat, lon)                      # [0] = 오늘, [1] = 내일
    rows.append({'도시': city, '날씨': WMO.get(d['weather_code'][1], '기타'),
                 '최저': round(d['temperature_2m_min'][1]), '오늘 최저': round(d['temperature_2m_min'][0]),
                 '최고': round(d['temperature_2m_max'][1]), '강수확률(%)': d['precipitation_probability_max'][1],
                 '강수량(mm)': d['precipitation_sum'][1], '최대풍속(m/s)': round(d['wind_speed_10m_max'][1])})
date = d['time'][1]
table = pd.DataFrame(rows).set_index('도시')
memo = f'[{date} 내일 날씨 예보 · 출처: Open-Meteo]\n' + '\n'.join(
    f"{c}: {r['날씨']}, 아침 최저 {r['최저']}도(오늘 {r['오늘 최저']}도), 낮 최고 {r['최고']}도, "
    f"강수확률 {r['강수확률(%)']}%, 예상 강수량 {r['강수량(mm)']}mm, 최대 풍속 초속 {r['최대풍속(m/s)']}m"
    for c, r in table.iterrows())
print(memo)
table

[2026-11-13 내일 날씨 예보 · 출처: Open-Meteo]
서울: 흐림, 아침 최저 8도(오늘 6도), 낮 최고 16도, 강수확률 7%, 예상 강수량 0.0mm, 최대 풍속 초속 1m
강릉: 흐림, 아침 최저 9도(오늘 6도), 낮 최고 17도, 강수확률 0%, 예상 강수량 0.0mm, 최대 풍속 초속 3m
대전: 흐림, 아침 최저 6도(오늘 3도), 낮 최고 17도, 강수확률 27%, 예상 강수량 0.0mm, 최대 풍속 초속 2m
부산: 이슬비, 아침 최저 9도(오늘 9도), 낮 최고 17도, 강수확률 5%, 예상 강수량 0.4mm, 최대 풍속 초속 2m
제주: 이슬비, 아침 최저 15도(오늘 11도), 낮 최고 18도, 강수확률 34%, 예상 강수량 4.3mm, 최대 풍속 초속 4m


,날씨,최저,오늘 최저,최고,강수확률(%),강수량(mm),최대풍속(m/s)
도시,,,,,,,
서울,흐림,8,6,16,7,0.0,1
강릉,흐림,9,6,17,0,0.0,3
대전,흐림,6,3,17,27,0.0,2
부산,이슬비,9,9,17,5,0.4,2
제주,이슬비,15,11,18,34,4.3,4


In [5]:
meteo = pd.DataFrame([{'title': f'{date} 내일 날씨 (Open-Meteo)', 'input': memo,
                       'instruction': '아래 날씨 예보 메모를 읽고, 앵커가 읽을 내일 날씨 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.'}])
ans_meteo = ask_models(MODELS, meteo, 'weather_meteo')
for n, a in ans_meteo.items():
    print(f'■ {n}\n{a[0]}\n')
make_report(meteo, ans_meteo, 'meteo_report.html', '스포츠로 배운 모델 × 날씨 데이터',
            note='정답이 없어요. 메모의 숫자 · 도시와 하나씩 대조하세요.', gold_col='-', src_label='날씨 메모')

묻는 중: 원본 Qwen2.5-3B (1문제)
묻는 중: 내가 학습한 모델 (1문제)
묻는 중: gpt-4.1-mini (1문제)
✅ 기록 저장: results\weather_meteo_20261112_1600.json
■ 원본 Qwen2.5-3B
안녕하세요, MBC 뉴스의 앵커입니다. 오늘은 기상 조사실에서 제공하는 내일의 날씨 예보에 대해 말씀드리겠습니다. 

우선, 서울에서는 아침에 시작되는 흐린 날씨가 계속될 것으로 보이며, 아침 최저 기온은 8도로 예상됩니다. 이는 오늘의 6도보다 약간 높은 수준입니다. 낮에는 최고 기온이 16도까지 오를 것으로 예상되며, 강수확률은 7%로 매우 낮은 수준입니다. 이는 대부분의 사람들이 걱정할 만한 강수 확률이 아닙니다.

강릉에서는 아침 최저 기온이 9도로, 서울보다 1도 높은 것으로 예상됩니다. 낮 최고 기온은 17도로, 서울과 비슷한 수준입니다. 강수확률은 0%로, 이는 내일 강수 없음을 의미합니다. 최대 풍속은 초속 3m로, 강风吹拂을 걱정할 필요는 없습니다.

대전에서는 아침 최저 기온이 6도로, 서울보다 2도 낮은 것으로 예상됩니다. 낮 최고 기온은 17도로, 서울과 비슷한 수준입니다. 강수확률은 27%로, 약간의 비가 내릴 가능성이 있습니다. 예상 강수량은 0.0mm로, 매우 적은 양입니다. 최대 풍속은 초속 2m로, 강风吹拂을 걱정할 필요는 없습니다.

부산에서는 아침 최저 기온이 9도로, 오늘과 동일한 수준입니다. 낮 최고 기온은 17도로, 서울과 비슷한 수준입니다. 강수확률은 5%로, 약간의 비가 내릴 가능성이 있습니다. 예상 강수량은 0.4mm로, 매우 적은 양입니다. 최대 풍속은 초속 2m로, 강风吹拂을 걱정할 필요는 없습니다.

제주에서는 아침 최저 기온이 15도로, 오늘보다 4도 높은 것으로 예상됩니다. 낮 최고 기온은 18도로, 서울과 비슷한 수준입니다. 강수확률은 34%로, 약간의 비가 내릴 가능성이 있습니다. 예상 강수량은 4.3mm로, 적은 양입니다. 최대 풍속은 초속 4m로, 강风吹拂

## 정리 — 무엇을 확인했나
| 질문 | 어디서 보나 |
|---|---|
| 처음 보는 주제에도 앵커 리드 형식이 나오나? | ① 끝 문장 · 길이 · "~는데요" |
| 지시문보다 학습한 습관이 강한가? | ② 평균 글자 수를 ①과 비교 |
| 숫자만 줘도 사실대로 쓰나? | ③ 메모와 대조 |

내 모델이 배운 것은 **형식**이에요. 사실이 맞는지는 언제나 사람이 확인해요.

⚠ `test_weather.jsonl`, `results/`, `*.html`에는 MBC 기사 내용이 들어 있어요. GitHub에 올리지 않아요.